# Techsweep text to .mat

Converts an ngspice techsweep of one device into the pygmid lookup table `<device>.mat`.

In [4]:
import numpy as np, pandas as pd
from scipy.io import savemat, loadmat

In [5]:
choice = 2
devices = ['nfet_01v8', 'pfet_01v8', 'pfet_01v8_lvt']
w = np.array([5])
nfing = np.array([1])
pdk_info = 'sky130A via open_pdks 1689ac3f2dc763876eaf967227c7dfe831b031ae, IIC-OSIC-TOOLS 2026.09, ngspice-47'

In [6]:
df_raw = pd.read_csv('techsweep_sky130A_'+devices[choice]+'.txt', sep=r"\s+")
par_names = df_raw.columns.to_list()
par_prefix = par_names[1].split('[')[0]

In [7]:
df = df_raw.drop(['frequency', 'frequency.1'], axis=1)
df = df.apply(pd.to_numeric)
df.columns = df.columns.str.removeprefix(par_prefix+'[')
df.columns = df.columns.str.replace(par_prefix[1:], '')
df.columns = df.columns.str.removesuffix(']')
df['l'] = df['l'].apply(lambda x: round(x*1e6, 3))
df

,cdd,cgb,cgd,cgdo,cgg,cgs,cgso,css,gds,gm,gmbs,id,l,vdsat,vth,onoise..1overf,onoise..id
0,1.749000e-16,-3.704000e-15,-1.531000e-16,1.000000e-16,3.804000e-15,5.343000e-17,1.000000e-16,-1.876000e-17,1.388000e-08,-1.860000e-27,-4.864000e-28,0.000000,0.35,0.04315,0.3361,0.000000e+00,8.976000e-15
1,4.326000e-17,-3.085000e-15,-3.988000e-17,1.000000e-16,3.113000e-15,1.143000e-17,1.000000e-16,-4.230000e-18,6.971000e-10,0.000000e+00,-2.551000e-28,0.000000,0.35,0.04533,0.4370,0.000000e+00,2.041000e-15
2,1.313000e-17,-2.695000e-15,-1.249000e-17,1.000000e-16,2.704000e-15,2.950000e-18,1.000000e-16,-1.148000e-18,5.147000e-11,0.000000e+00,0.000000e+00,0.000000,0.35,0.04670,0.5181,0.000000e+00,1.971000e-16
3,4.573000e-18,-2.422000e-15,-4.449000e-18,1.000000e-16,2.426000e-15,8.250000e-19,1.000000e-16,-3.760000e-19,5.001000e-12,0.000000e+00,0.000000e+00,0.000000,0.35,0.04769,0.5877,0.000000e+00,1.915000e-17
4,2.552000e-18,-2.294000e-15,-2.511000e-18,1.000000e-16,2.296000e-15,3.591000e-19,1.000000e-16,-2.029000e-19,1.364000e-12,0.000000e+00,-2.560000e-31,0.000000,0.35,0.04818,0.6256,0.000000e+00,5.222000e-18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
266445,1.249000e-16,1.032000e-14,-9.430000e-17,1.000000e-16,3.077000e-13,-3.180000e-13,1.000000e-16,2.011000e-13,1.184000e-07,3.961000e-05,7.864000e-06,0.000028,10.00,1.08300,0.4479,5.301000e-11,7.236000e-13
266446,1.062000e-16,-3.919000e-15,-8.816000e-17,1.000000e-16,2.987000e-13,-2.947000e-13,1.000000e-16,1.888000e-13,1.015000e-07,3.375000e-05,6.629000e-06,0.000024,10.00,1.02600,0.5818,5.121000e-11,6.425000e-13
266447,8.851000e-17,-9.721000e-15,-7.793000e-17,1.000000e-16,2.924000e-13,-2.826000e-13,1.000000e-16,1.817000e-13,8.908000e-08,2.975000e-05,5.475000e-06,0.000021,10.00,0.96940,0.6878,4.904000e-11,5.827000e-13
266448,7.375000e-17,-1.270000e-14,-6.774000e-17,1.000000e-16,2.874000e-13,-2.746000e-13,1.000000e-16,1.769000e-13,8.002000e-08,2.689000e-05,4.558000e-06,0.000019,10.00,0.91640,0.7769,4.685000e-11,5.386000e-13


In [8]:
vgs = np.arange(0, 1.8001, 0.025)
vds = np.arange(0, 1.8001, 0.025)
vsb = np.array([0, 0.5, 1.0, 1.5, 1.8])
l = np.unique(abs(df['l']))
shape = (len(l), len(vgs), len(vds), len(vsb))

def grab(name): return df[name].to_numpy().reshape(shape)

dic = {
  "INFO": f"sky130A {devices[choice]}, BSIM4; {pdk_info}", "CORNER": "NOM", "TEMP": 300,
  "VGS": vgs, "VDS": vds, "VSB": vsb, "L": l, "W": float(w[0]) * 1e-6, "NFING": float(nfing[0]),
  "ID": grab("id"), "VT": grab("vth"), "VDSAT": grab("vdsat"), "GM": grab("gm"), "GMB": grab("gmbs"),
  "GDS": grab("gds"), "CGG": grab("cgg"), "CGB": grab("cgb"), "CGD": grab("cgd"),
  "CGS": grab("cgs"), "CDD": grab("cdd"), "CSS": grab("css"), "CGSO": grab("cgso"), "CGDO": grab("cgdo"),
  "STH": grab("onoise..id")**2, "SFL": grab("onoise..1overf")**2,
}

savemat(devices[choice]+'.mat', {devices[choice]: dic})